In [1]:
from pathlib import Path

import joblib
import pandas as pd

from src.config import TARGET_COL, PROCESSED_DATA_DIR, MODELS_DIR
from src.data import load_train, save_processed
from src.features import TitanicFeatures
from src.modeling import build_model
from src.evaluate import cv_scores
from src.logging_utils import log_experiment
from src.feature_search import (
    run_openfe, apply_openfe, save_features,
    feature_importance_report, ablation_openfe
)

In [2]:
def run_experiment(
    model_name: str = "logreg",
    params: dict | None = None,
    run_id: str | None = None,
    save_model: bool = True,
) -> tuple[float, float]:
    """Полный цикл: загрузка → фичи → модель → CV → лог → fit → сохранение."""
    # 1. Загрузка
    df = load_train()
    X_raw = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL]

    # 2. Фичи
    fe = TitanicFeatures()
    X = fe.transform(X_raw)

    # 3. Модель
    model = build_model(model_name, X, params=params)

    # 4. CV
    mean, std, scores = cv_scores(model, X, y)

    return mean, std

In [3]:
mean, std = run_experiment()

In [4]:
print(f"CV {'logreg'}: mean={mean} std={std}")

CV logreg: mean=0.836137091205825 std=0.004268038251999833


In [5]:
df = load_train()
X_raw = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

In [6]:
model = build_model('logreg', X_raw, params=None)

mean, std, scores = cv_scores(model, X_raw, y)

In [7]:
print(f"CV {'logreg'}: mean={mean} std={std}")

CV logreg: mean=0.8069612704789403 std=0.0332360185803288


In [8]:
fe = TitanicFeatures()
X = fe.transform(X_raw)

# 2. Запуск OpenFE (1-2 минуты)
features = run_openfe(X, y, n_features=30)

# 3. Смотрим что сгенерировано
report = feature_importance_report(features, top_n=20)
print(report.to_string(index=False))

The number of candidate features is 602
Start stage I selection.


100%|██████████| 4/4 [00:12<00:00,  3.01s/it]


343 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|██████████| 4/4 [00:08<00:00,  2.01s/it]


The number of remaining candidate features is 250
Start stage II selection.


100%|██████████| 4/4 [00:04<00:00,  1.19s/it]


Finish data processing.
[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002517 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10470
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 256
 rank        operator feature
    1               - unknown
    2  GroupByThenStd unknown
    3               - unknown
    4 GroupByThenRank unknown
    5 GroupByThenRank unknown
    6               + unknown
    7               / unknown
    8               / unknown
    9               * unknown
   10 GroupByThenRank unknown
   11               + unknown
   12 GroupByThenRank unknown
   13            freq unknown
   14               / unknown
   15 GroupByThenRank unknown
   16 GroupByThenRank unknown
   17  GroupByThenMax unknown
   18 GroupByThenRank unknown
   19  GroupByThenStd unknown
   20  GroupByThenSt

In [9]:
from openfe import transform

# X — это твой текущий X после TitanicFeatures.transform(...)
X_with_ofe, _ = transform(X, X, features, n_jobs=1)  # test можно временно тоже X

base_cols = set(X.columns)
all_cols  = list(X_with_ofe.columns)

new_cols = [c for c in all_cols if c not in base_cols]
print("New OFE features:")
for c in new_cols:
    print(c)

New OFE features:
autoFE_f_0
autoFE_f_1
autoFE_f_2
autoFE_f_3
autoFE_f_4
autoFE_f_5
autoFE_f_6
autoFE_f_7
autoFE_f_8
autoFE_f_9
autoFE_f_10
autoFE_f_11
autoFE_f_12
autoFE_f_13
autoFE_f_14
autoFE_f_15
autoFE_f_16
autoFE_f_17
autoFE_f_18
autoFE_f_19
autoFE_f_20
autoFE_f_21
autoFE_f_22
autoFE_f_23
autoFE_f_24
autoFE_f_25
autoFE_f_26
autoFE_f_27
autoFE_f_28
autoFE_f_29


In [10]:
from openfe import tree_to_formula

mapping = {}
for feat, col in zip(features, new_cols):
    try:
        mapping[col] = tree_to_formula(feat)  # человекочитаемая формула
    except Exception:
        mapping[col] = f"{feat.name}<?>"

In [11]:
for col in new_cols:
    print(col, "→", mapping[col])

autoFE_f_0 → (Sex-autoFE_f_5_manual)
autoFE_f_1 → GroupByThenStd(familysize,Pclass_Sex)
autoFE_f_2 → (Pclass-Pclass_Sex)
autoFE_f_3 → GroupByThenRank(familysize,Title)
autoFE_f_4 → GroupByThenRank(Fare,autoFE_f_5_manual)
autoFE_f_5 → (Age+familysize)
autoFE_f_6 → (Sex/familysize)
autoFE_f_7 → (Fare/familysize)
autoFE_f_8 → (Age*familysize)
autoFE_f_9 → GroupByThenRank(Fare,Pclass)
autoFE_f_10 → (Fare+Age)
autoFE_f_11 → GroupByThenRank(Fare,Age)
autoFE_f_12 → freq(Fare)
autoFE_f_13 → (Fare/Pclass)
autoFE_f_14 → GroupByThenRank(Age,autoFE_f_5_manual)
autoFE_f_15 → GroupByThenRank(Fare,Embarked)
autoFE_f_16 → GroupByThenMax(Fare,familysize)
autoFE_f_17 → GroupByThenRank(Fare,familysize)
autoFE_f_18 → GroupByThenStd(Pclass_Sex,Age)
autoFE_f_19 → GroupByThenStd(autoFE_f_5_manual,Age)
autoFE_f_20 → GroupByThenNUnique(Title,Age)
autoFE_f_21 → GroupByThenStd(Fare,Age)
autoFE_f_22 → GroupByThenRank(Age,Pclass_Sex)
autoFE_f_23 → (Age*autoFE_f_5_manual)
autoFE_f_24 → GroupByThenMean(Fare,familysi

In [12]:
base_score, base_std, _ = cv_scores(build_model("logreg", X), X, y)

In [13]:
results = []

for col in new_cols:
    X_exp = pd.concat([X, X_with_ofe[[col]]], axis=1)
    mean, std, _ = cv_scores(build_model("logreg", X_exp), X_exp, y)
    results.append({
        "col":     col,
        "formula": mapping.get(col, col),
        "mean":    round(mean, 5),
        "std":     round(std, 5),
        "delta":   round(mean - base_score, 5),
    })

df_results = pd.DataFrame(results).sort_values("delta", ascending=False)
print(df_results.to_string(index=False))

        col                                  formula    mean     std    delta
autoFE_f_21                 GroupByThenStd(Fare,Age) 0.83951 0.00442  0.00337
autoFE_f_23                  (Age*autoFE_f_5_manual) 0.83950 0.00930  0.00336
autoFE_f_13                            (Fare/Pclass) 0.83837 0.00682  0.00223
autoFE_f_12                               freq(Fare) 0.83726 0.00616  0.00112
autoFE_f_15           GroupByThenRank(Fare,Embarked) 0.83724 0.01567  0.00110
 autoFE_f_5                         (Age+familysize) 0.83614 0.00427  0.00000
autoFE_f_18           GroupByThenStd(Pclass_Sex,Age) 0.83614 0.00427  0.00000
 autoFE_f_2                      (Pclass-Pclass_Sex) 0.83614 0.00427  0.00000
autoFE_f_27                               (Fare/Sex) 0.83614 0.00427  0.00000
autoFE_f_25 CombineThenFreq(Title,autoFE_f_5_manual) 0.83614 0.00969  0.00000
 autoFE_f_0                  (Sex-autoFE_f_5_manual) 0.83613 0.00674 -0.00001
autoFE_f_17         GroupByThenRank(Fare,familysize) 0.83613 0.0

In [14]:


# Сортированная таблица результатов (уже есть у тебя как df_results)
# Оставляем только те что delta > 0

groups = {
    "delta >= 0.005": df_results[df_results["delta"] >= 0.005]["col"].tolist(),
    "delta >= 0.002": df_results[df_results["delta"] >= 0.002]["col"].tolist(),
    "delta >= 0.001": df_results[df_results["delta"] >= 0.001]["col"].tolist(),
    "delta > 0":      df_results[df_results["delta"] >  0.000]["col"].tolist(),
}

base_score, base_std, _ = cv_scores(build_model("logreg", X), X, y)
print(f"Baseline: {base_score:.5f} ± {base_std:.5f}\n")

rows = [{"threshold": "baseline", "n_features": 0, "mean": base_score, "std": base_std, "delta": 0.0, "cols": []}]

for threshold_name, cols in groups.items():
    X_exp = pd.concat([X, X_with_ofe[cols]], axis=1)
    mean, std, _ = cv_scores(build_model("logreg", X_exp), X_exp, y)
    delta = mean - base_score
    rows.append({
        "threshold":  threshold_name,
        "n_features": len(cols),
        "mean":       round(mean, 5),
        "std":        round(std, 5),
        "delta":      round(delta, 5),
        "cols":       cols,
    })
    print(f"{threshold_name:20s}  n={len(cols):2d}  mean={mean:.5f}  delta={delta:+.5f}")

comparison_df = pd.DataFrame(rows).drop(columns="cols")
print("\n", comparison_df.to_string(index=False))

Baseline: 0.83614 ± 0.00427

delta >= 0.005        n= 0  mean=0.83614  delta=+0.00000
delta >= 0.002        n= 3  mean=0.84062  delta=+0.00448
delta >= 0.001        n= 5  mean=0.83501  delta=-0.00112
delta > 0             n= 5  mean=0.83501  delta=-0.00112

      threshold  n_features     mean      std    delta
      baseline           0 0.836137 0.004268  0.00000
delta >= 0.005           0 0.836140 0.004270  0.00000
delta >= 0.002           3 0.840620 0.007800  0.00448
delta >= 0.001           5 0.835010 0.016920 -0.00112
     delta > 0           5 0.835010 0.016920 -0.00112


In [15]:
sorted_feats = df_results.sort_values("delta", ascending=False)

current_cols = []
current_X = X.copy()
current_score, current_std, _ = cv_scores(build_model("logreg", current_X), current_X, y)
threshold = 0.002

history = []
for _, row in sorted_feats.iterrows():
    col = row["col"]
    cand_X = pd.concat([current_X, X_with_ofe[[col]]], axis=1)
    mean, std, _ = cv_scores(build_model("logreg", cand_X), cand_X, y)
    delta = mean - current_score

    history.append((col, mean, std, delta))

    if delta >= threshold:
        print(f"KEEP {col:10s} mean={mean:.5f} delta={delta:+.5f}")
        current_X = cand_X
        current_cols.append(col)
        current_score = mean
    else:
        print(f"DROP {col:10s} mean={mean:.5f} delta={delta:+.5f}")

print("\nFinal score:", current_score)
print("Selected features:", current_cols)

KEEP autoFE_f_21 mean=0.83951 delta=+0.00337
DROP autoFE_f_23 mean=0.84062 delta=+0.00111
DROP autoFE_f_13 mean=0.84063 delta=+0.00112
DROP autoFE_f_12 mean=0.83727 delta=-0.00224
DROP autoFE_f_15 mean=0.83501 delta=-0.00450
DROP autoFE_f_5 mean=0.83838 delta=-0.00112
DROP autoFE_f_18 mean=0.83614 delta=-0.00336
DROP autoFE_f_2 mean=0.83838 delta=-0.00112
DROP autoFE_f_27 mean=0.83727 delta=-0.00224
DROP autoFE_f_25 mean=0.83950 delta=-0.00001
DROP autoFE_f_0 mean=0.83838 delta=-0.00113
DROP autoFE_f_17 mean=0.83614 delta=-0.00336
DROP autoFE_f_10 mean=0.83614 delta=-0.00336
DROP autoFE_f_3 mean=0.84063 delta=+0.00112
DROP autoFE_f_20 mean=0.83501 delta=-0.00449
DROP autoFE_f_19 mean=0.83502 delta=-0.00449
DROP autoFE_f_7 mean=0.83503 delta=-0.00448
DROP autoFE_f_14 mean=0.83727 delta=-0.00224
DROP autoFE_f_4 mean=0.83500 delta=-0.00451
DROP autoFE_f_9 mean=0.83726 delta=-0.00225
DROP autoFE_f_29 mean=0.83389 delta=-0.00562
DROP autoFE_f_28 mean=0.83614 delta=-0.00337
DROP autoFE_f_22 

In [16]:
current_X.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Pclass             891 non-null    int64  
 1   Sex                891 non-null    int64  
 2   Age                714 non-null    float64
 3   Fare               891 non-null    float64
 4   Embarked           889 non-null    str    
 5   Title              891 non-null    str    
 6   familysize         891 non-null    int64  
 7   Pclass_Sex         891 non-null    int64  
 8   autoFE_f_5_manual  889 non-null    float64
 9   autoFE_f_21        714 non-null    float64
dtypes: float64(4), int64(4), str(2)
memory usage: 73.1 KB


In [17]:
current_X["autoFE_f_1"] = current_X["autoFE_f_1"].fillna(0.0)

KeyError: 'autoFE_f_1'

In [18]:
mean, std, _ = cv_scores(build_model("logreg", current_X), current_X, y)

In [19]:
print(f"mean={mean} std={std}")

mean=0.8395078777226791 std=0.004417451959921653


In [22]:
for item in history:
    print(item)

('autoFE_f_1', 0.8338899001945892, 0.00464319062602027, 0.0033645094469900716)
('autoFE_f_5', 0.8361308141359614, 0.006739228952984528, 0.002240913941372158)
('autoFE_f_12', 0.8338836231247253, 0.009306238219988322, -0.0022471910112360494)
('autoFE_f_7', 0.8316427091833531, 0.007257861860353865, -0.004488104952608207)
('autoFE_f_15', 0.8305191136777352, 0.0067461711434545595, -0.005611700458226121)
('autoFE_f_6', 0.8327663046889711, 0.00836313501942534, -0.0033645094469902936)
('autoFE_f_9', 0.8361370912058252, 0.007489336174046487, 6.277069863891427e-06)
('autoFE_f_10', 0.8338961772644529, 0.00908946159349088, -0.0022346368715084886)
('autoFE_f_11', 0.8350072186303434, 0.00994161841504319, -0.0011235955056179137)
('autoFE_f_8', 0.8316489862532169, 0.013766331204013917, -0.004481827882744427)
('autoFE_f_18', 0.8338899001945892, 0.010484841410660747, -0.002240913941372158)
('autoFE_f_17', 0.8338899001945892, 0.00464319062602027, -0.002240913941372158)
('autoFE_f_14', 0.8327600276191074,

In [23]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# 1. Базовый список признаков (без autoFE_f_1, autoFE_f_5)
base_features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Title", "familysize"]

# 2. Добавляем ручной признак Pclass_Sex
current_X["Pclass_Sex"] = current_X["Pclass"] * current_X["Sex"]

# 3. Варианты наборов колонок
feature_sets = {
    "baseline": base_features,
    "baseline + Pclass_Sex": base_features + ["Pclass_Sex"],
    "baseline + Pclass_Sex + autoFE_f_5": base_features + ["Pclass_Sex", "autoFE_f_5"],
    "baseline + autoFE_f_1 + autoFE_f_5": base_features + ["autoFE_f_1", "autoFE_f_5"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, feats in feature_sets.items():
    X_cur = current_X[feats].copy()

    mean, std, _ = cv_scores(build_model("logreg", current_X, params={"max_iter":1000,'C':1.0}), current_X, y)

    print(f"{name}: mean={mean}, std={std}")

baseline: mean=0.836137091205825, std=0.004268038251999833
baseline + Pclass_Sex: mean=0.836137091205825, std=0.004268038251999833
baseline + Pclass_Sex + autoFE_f_5: mean=0.836137091205825, std=0.004268038251999833
baseline + autoFE_f_1 + autoFE_f_5: mean=0.836137091205825, std=0.004268038251999833


In [24]:
mean, std, _ = cv_scores(build_model("logreg", current_X), current_X, y)
print(f"mean={mean} std={std}")

mean=0.8361308141359614 std=0.006739228952984528


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.base import clone
from scipy.stats import loguniform
import numpy as np

def tune_with_random_search(
    base_model,
    X,
    y,
    param_distributions,
    n_iter=30,
    cv=None,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=1,
):
    """
    base_model        — Pipeline/модель, уже собранная (prep + clf)
    param_distributions — словарь с распределениями гиперпараметров (для RandomizedSearchCV)
    возвращает: best_model, best_params, search_obj
    """
    if cv is None:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring=scoring,
        cv=cv,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        refit=True,  # чтобы сразу обучить лучшую модель на всех данных
    )

    search.fit(X, y)
    best_model = search.best_estimator_
    best_params = search.best_params_
    print(f"RandomizedSearchCV best score: {search.best_score_}")
    print(f"Best params: {best_params}")

    return best_model, best_params, search

In [ ]:
# pipeline = Pipeline([("prep", preprocessor), ("clf", LogisticRegression(max_iter=1000))])

param_distributions = {
    "model__C": loguniform(1e-3, 1e3),
    "model__solver": ["lbfgs", "liblinear"],
}

best_model_rs, best_params_rs, rs_obj = tune_with_random_search(
    base_model=build_model("logreg", current_X, params={"max_iter":1000,'C':1.0}),
    X=current_X,
    y=y,
    param_distributions=param_distributions,
    n_iter=50,
)

In [29]:
from src.tuning import tune_with_random_search
_, best_params, search = tune_with_random_search("logreg", current_X, y)
print(f"RandomizedSearchCV best score: {search.best_score_}")
print(f"Best params: {best_params}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomizedSearchCV best score: 0.8361308141359614
Best params: {'model__C': np.float64(3.907967156822881), 'model__solver': 'lbfgs'}


In [46]:
import importlib
import src.tuning_objectives
import src.tuning

importlib.reload(src.tuning_objectives)
importlib.reload(src.tuning)

<module 'src.tuning' from 'C:\\Users\\Barenin Vitalii\\PycharmProjects\\kaggle-titanic\\src\\tuning.py'>

In [35]:
from src.tuning import tune_with_optuna
best_model, study = tune_with_optuna("logreg", current_X, y)
print(f"Optuna best score: {study.best_value}")
print(f"Best params: {study.best_trial.params}")

Optuna best score: 0.8372544096415793
Best params: {'model__C': 0.9550397388841476, 'model__solver': 'lbfgs'}


In [47]:
importlib.reload(src.features)
importlib.reload(src.config)
from src.features import TitanicFeatures
from src.modeling import build_model

In [49]:
fe = TitanicFeatures()
X = fe.transform(X_raw)
model = build_model('logreg', X)

mean, std, scores = cv_scores(model, X, y)
print(f"CV {'logreg'}: mean={mean} std={std}")

CV logreg: mean=0.8372669637813068 std=0.004824449889940526


In [21]:
current_X.head()

,Pclass,Sex,Age,Fare,Embarked,Title,familysize,Pclass_Sex,autoFE_f_5_manual,autoFE_f_21
openfe_index,,,,,,,,,,
0,3,0,22.0,7.2500,S,Mr,2,0,0.726708,37.655129
1,1,1,38.0,71.2833,C,Mrs,2,1,0.255952,70.996756
2,3,1,26.0,7.9250,S,Miss,1,3,0.726708,19.057277
3,1,1,35.0,53.1000,S,Mrs,2,1,0.099379,155.599330
4,3,0,35.0,8.0500,S,Mr,1,0,0.726708,155.599330


In [50]:
X.head()

,Pclass,Sex,Age,Fare,Embarked,Title,familysize,Pclass_Sex,autoFE_f_21_manual
0,3,0,22.0,7.2500,S,Mr,2,0,38.015474
1,1,1,38.0,71.2833,C,Mrs,2,1,72.750026
2,3,1,26.0,7.9250,S,Miss,1,3,19.335500
3,1,1,35.0,53.1000,S,Mrs,2,1,157.870974
4,3,0,35.0,8.0500,S,Mr,1,0,157.870974


In [45]:
X[X.select_dtypes(include="number").columns].corr()

,Pclass,Sex,Age,Fare,familysize,Pclass_Sex,autoFE_f_5_manual,autoFE_f_21_manual
Pclass,1.000000,-0.131900,-0.369226,-0.549500,0.065997,0.150380,0.940926,-0.160011
Sex,-0.131900,1.000000,-0.093254,0.182333,0.200988,0.897070,-0.137331,0.061837
Age,-0.369226,-0.093254,1.000000,0.096067,-0.301914,-0.186382,-0.359046,0.135984
Fare,-0.549500,0.182333,0.096067,1.000000,0.217138,-0.030576,-0.453775,0.261057
familysize,0.065997,0.200988,-0.301914,0.217138,1.000000,0.229815,0.077768,-0.094713
Pclass_Sex,0.150380,0.897070,-0.186382,-0.030576,0.229815,1.000000,0.113459,-0.002056
autoFE_f_5_manual,0.940926,-0.137331,-0.359046,-0.453775,0.077768,0.113459,1.000000,-0.142560
autoFE_f_21_manual,-0.160011,0.061837,0.135984,0.261057,-0.094713,-0.002056,-0.142560,1.000000
